# {{场景}}指标验证 — 每个数都能自己核

一节算一个指标，每节同一个节奏：**口径一句话 → 怎么算的 → 算出来长什么样 → 和什么对上了账**。
每节都给了核对路径（哪张表、什么条件、抽哪几笔），欢迎亲手核。

- 数字可信 → 写进「你的结论」；觉得口径不对 → 回方案文档改口径（会记档），这里不悄悄绕。
- 全部确认后计算会做成正式脚本，这一步的数字就是脚本的对照标准。

In [ ]:
# 接第三步确认的清洗结果（或直接 read + 清洗链）
clean = {{清洗链或读取}}

pd.DataFrame([{
    "行数": f"{len(clean):,}",
    "金额合计(万)": round(float(clean["{{金额列}}"].sum()), 1),
    "期间": "{{数据覆盖的起止月份}}",
}])

## 二、{{指标名}}

**口径**：{{一句话大白话，说什么算什么}}　**粒度**：{{一行 = 一个物料×一个月}}，加总 = {{什么}}

In [ ]:
# 先看中间量：{{例：逐月逐物料的量、价}}
mid = clean.groupby(["{{维度列}}", "{{期间列}}"]).agg({{"{{量列}}": "sum"}})
mid.tail(12)

In [ ]:
# 按 1.4 确认的公式算出指标
result = {{公式}}
result.head(15)

In [ ]:
# 对账三件套，一张表摆平：对不上必须解释差异来源
pd.DataFrame([
    {"对什么": "业务已知数（{{1.3 的报表/口头数}}）", "参照数": {{x}}, "本表": {{y}}, "差异": "{{有则写来源}}"},
    {"对什么": "抽样手算 {{n}} 笔",                  "参照数": {{x}}, "本表": {{y}}, "差异": "{{...}}"},
    {"对什么": "分层加总：明细→分组→总计",            "参照数": {{x}}, "本表": {{y}}, "差异": "{{...}}"},
])

### 这张报表长这样（未来 `data/outputs/` 里的那张表）

一行 = {{一个物料×一个月}}，加总 = {{...}}。列、单位、粒度有不合适的，就在这格说，改了重跑本节。

In [ ]:
# 输出表候选形态：列 + 单位 + 粒度，全量 dataframe 交给业务翻（不截断）
report = {{报表 df}}                      # 一行 = {{粒度}}

cols = pd.DataFrame({"列": report.columns, "示例": [report[c].iloc[0] for c in report.columns]})
cols["口径/单位"] = ["{{逐列填}}"] * len(cols)
cols


**你的结论**：{{数字可信 / 口径要改（回 1.4，记 4.2）}} —— 定了再看下一个指标

---

{{「一节」照此复制，一个指标一份，编号往后排}}

## 最后、和正式脚本对拍（脚本沉淀完才跑）

同一窗口跑一遍正式脚本，结果和本 notebook 逐数比对。

In [ ]:
script = {{脚本产出的结果表}}

pd.DataFrame([{
    "{{指标/物料}}": {{键}},
    "notebook": {{本表值}},
    "脚本": {{脚本值}},
    "一致": {{是否一致}},
}])   # 全部「一致」才算落地

**待确认事项**：{{逐条}}；口径变更已记入 4.2。

落地后本 notebook 保留作对照标准；换数据重跑时先跑脚本，再抽本 notebook 对拍。

## 输出表定稿（每张确认过的表记一张卡，抄进方案 4.3）

表清单 = 下面这几张在 notebook 里被业务点头的报表，不再另起一套字段名。

In [ ]:
pd.DataFrame([
    {"表名": "{{表1}}", "粒度(主键)": "{{物料×月份}}", "列": "{{列1, 列2, ...}}",
     "更新方式": "全量重算", "看板取": "KPI / 趋势 / 下钻",
     "归档": "data/outputs/{{场景}}/2026/2026-09/2026-09-30/"},
    {"表名": "{{表2}}", "粒度(主键)": "{{...}}", "列": "{{...}}",
     "更新方式": "{{...}}", "看板取": "{{...}}", "归档": "{{...}}"},
])

## 落地、归档（脚本确认后才跑）

同一窗口跑一遍正式脚本，结果落成两份：`data.parquet` 给复算对拍、`data.json` 给看板取数。
两份由**一次运行**同时产出，不手工补；路径形如
`data/outputs/{{场景}}/2026/2026-09/2026-09-30/`。

In [ ]:
from reporting.metrics import archive_metrics

paths = archive_metrics(
    {{主表 df}},                      # 一行一业务粒度（4.1 验证过的指标）
    scene="{{场景}}",                 # 与 notebooks/<场景>/ 同名
    snapshot_date="{{业务结束日}}",    # YYYY-MM-DD，业务结束日，不是跑数日
    tables={
        "{{月度块}}": {{月度 df}},
        "{{汇总块}}": {{汇总 df}},
    },
    meta={
        "window_start": "{{...}}", "window_end": "{{...}}",
        "data_file": {{取数 meta 里的数据文件}},
        "entrypoint": "{{uv run python scripts/xxx.py}}",
    },
)
paths

**两份对得上才算落地**：读回 parquet 和 json，行数、列名、金额合计逐项比 —— 对不上说明有一边落了旧数据。

In [ ]:
import json
from reporting.metrics import read_outputs

back, info = read_outputs("{{场景}}")
payload = json.loads(paths["json"].read_text(encoding="utf-8"))
detail = payload["tables"]["detail"]

pd.DataFrame([
    {"对什么": "行数", "parquet": len(back), "json": len(detail["rows"]),
     "一致": len(back) == len(detail["rows"])},
    {"对什么": "列名", "parquet": list(back.columns), "json": detail["columns"],
     "一致": list(back.columns) == detail["columns"]},
    {"对什么": "金额合计(万)", "parquet": round(float(back["{{金额列}}"].sum()), 1),
     "json": round(sum(r["{{金额列}}"] or 0 for r in detail["rows"]), 1),
     "一致": True},
])

**交付**：`paths["json"]` 就是第五步看板的输入。看板直接内嵌这份 json，不再算数、不再回读 parquet；
`paths["parquet"]` 用于复算与对拍。把两个路径抄进方案 4.3，然后进第五步。